In [1]:
import os
os.getcwd()

'C:\\Users\\TWH'

In [2]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, MinMaxScaler

In [3]:
# 1. Define weather classification lists
SEVERE_WEATHER = ['Thunderstorm', 'Squall', 'Snow']
LOW_VIS_WEATHER = ['Fog', 'Mist', 'Haze', 'Smoke']

In [4]:
# 2. Load raw dataset
df = pd.read_csv("data/raw/Metro_Interstate_Traffic_Volume.csv")

In [5]:
# 3. Clean basic data anomalies & deduplication
df = df.drop_duplicates().copy()
df['date_time'] = pd.to_datetime(df['date_time'])
df['weather_description'] = df['weather_description'].str.strip().str.lower()
df['weather_main'] = df['weather_main'].str.strip()
df['holiday'] = df['holiday'].fillna('None').str.strip()

# Impute temperature dropouts (0.0 K) with monthly median
df['month'] = df['date_time'].dt.month
for m in df['month'].unique():
    median_temp = df.loc[(df['month'] == m) & (df['temp'] >= 200.0), 'temp'].median()
    df.loc[(df['month'] == m) & (df['temp'] < 200.0), 'temp'] = median_temp

# Impute rain outliers (> 100mm) with monthly median
for m in df['month'].unique():
    median_rain = df.loc[(df['month'] == m) & (df['rain_1h'] <= 100.0), 'rain_1h'].median()
    df.loc[(df['month'] == m) & (df['rain_1h'] > 100.0), 'rain_1h'] = median_rain

df.drop(columns=['month'], inplace=True)

In [6]:
# 4. Cyclical time features
df['hour'] = df['date_time'].dt.hour
df['day_of_week'] = df['date_time'].dt.dayofweek  # Monday=0, Sunday=6

df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24.0)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24.0)
df['dow_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7.0)
df['dow_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7.0)

In [7]:
# 5. Binary flags & weather indicators
df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)
df['is_holiday'] = (df['holiday'] != 'None').astype(int)

df['is_severe_weather'] = df['weather_main'].isin(SEVERE_WEATHER).astype(int)
df['is_low_visibility'] = df['weather_main'].isin(LOW_VIS_WEATHER).astype(int)

In [8]:
# 6. Continuous feature scaling
scaler_std = StandardScaler()
scaler_minmax = MinMaxScaler()

df['temp_scaled'] = scaler_std.fit_transform(df[['temp']])
df['clouds_scaled'] = scaler_minmax.fit_transform(df[['clouds_all']])

In [9]:
# ==============================================================================
# 7. CONGESTION CATEGORY (Data-driven quartiles)
# ==============================================================================
q1, q2, q3 = df["traffic_volume"].quantile([0.25, 0.5, 0.75]).values

def bucket(v):
    if v <= q1:
        return "Low"
    elif v <= q2:
        return "Medium"
    elif v <= q3:
        return "High"
    return "Severe"

df["congestion_category"] = df["traffic_volume"].apply(bucket)

In [10]:
# ==============================================================================
# 8. PROXY ACCIDENT-RISK LABEL
# High risk = severe/low-visibility weather conditions occurring
# together with High/Severe congestion.
# ==============================================================================
high_congestion = df["congestion_category"].isin(["High", "Severe"])

risky_weather = (
    df["weather_main"].isin(SEVERE_WEATHER)
    | (df["is_low_visibility"] == 1)
)

df["high_risk"] = (high_congestion & risky_weather).astype(int)

In [11]:
# ==============================================================================
# 9. Save updated dataset to processed CSV
# ==============================================================================
os.makedirs("data/processed", exist_ok=True)
df.to_csv("data/processed/cleaned_traffic_features.csv", index=False)

print("Data processing pipeline complete!")
print(f"Total Rows: {len(df)} | Total Columns: {len(df.columns)}")
print("\nHigh Risk Proxy Distribution:")
print(df["high_risk"].value_counts())
print("\nPercentage High Risk:")
print(df["high_risk"].value_counts(normalize=True) * 100)

Data processing pipeline complete!
Total Rows: 48187 | Total Columns: 23

High Risk Proxy Distribution:
high_risk
0    42749
1     5438
Name: count, dtype: int64

Percentage High Risk:
high_risk
0    88.714799
1    11.285201
Name: proportion, dtype: float64
